# Lab02: Image Colorization

> Dataset structure:
> ```
> <LAB_ROOT>/
> ├── dataset/
> │   ├── test_public.npy
> │   ├── train_lab.npy
> │   ├── train_labels.npy
> │   ├── val_lab.npy
> │   └── val_laabels.npy
> ├── model/
> │   ├── __init__.py         # package init file
> │   └── cnn.py              # implement your CNN here
> ├── evaluation.py           # official metric - do not modify evaluation.py
> └── Lab02.ipynb       # this notebook file
> ```

## Mount Google Drive

In [ ]:
# --- Colab-only: mount your Google Drive ---
# (skipped automatically when running locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("not running on Colab, skip mounting")

In [ ]:
# Modify this part to the path where your code is located
import os
LAB_ROOT = "/content/drive/MyDrive/2026_DL_Lab02/"
if os.path.isdir(LAB_ROOT):
    os.chdir(LAB_ROOT)
print("working dir:", os.getcwd())
print(sorted(os.listdir(".")))

## Setup

In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import evaluation as E   # official metric - do not modify evaluation.py

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = Path("dataset")
print(f"device: {DEVICE}   torch: {torch.__version__}")

## Dataset

### Load dataset

In [ ]:
train_lab = np.load(DATA_DIR / "train_lab.npy")
val_lab = np.load(DATA_DIR / "val_lab.npy")

print(f"train {train_lab.shape}  val {val_lab.shape}  dtype {train_lab.dtype}")
print(f"L*  range [{train_lab[..., 0].min():6.1f}, {train_lab[..., 0].max():6.1f}]")
print(f"a*  range [{train_lab[..., 1].min():6.1f}, {train_lab[..., 1].max():6.1f}]")
print(f"b*  range [{train_lab[..., 2].min():6.1f}, {train_lab[..., 2].max():6.1f}]")
print(f"mean chroma (colour saturation) of the ground truth: {E.mean_chroma(train_lab[..., 1:]):.2f}")

In [ ]:
class ColorizationDataset(Dataset):

    def __init__(self, lab: np.ndarray, augment: bool = False):
        L, ab = E.normalize_lab(lab)
        self.L = torch.from_numpy(L)
        self.ab = torch.from_numpy(ab)
        self.augment = augment

    def __len__(self):
        return len(self.L)

    def __getitem__(self, i):
        L, ab = self.L[i], self.ab[i]
        if self.augment:
            # example augmentation: horizontal flip
            if torch.rand(1).item() < 0.5:
                L, ab = torch.flip(L, dims=[2]), torch.flip(ab, dims=[2])
            
            # ============================================================
            # more augmentation: none.  Colour statistics are what the model must
            # learn, so only the geometric flip above is used (see report).
            # ============================================================
        return L, ab

# ============================================================
# batch size
# ============================================================
BATCH_SIZE = 128
# ============================================================

train_set = ColorizationDataset(train_lab, augment=True)
val_set = ColorizationDataset(val_lab, augment=False)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2)

L0, ab0 = train_set[0]
print(f"{len(train_set)} train / {len(val_set)} val samples")
print(f"one sample: L {tuple(L0.shape)} in [{L0.min():.2f}, {L0.max():.2f}], "
      f"ab {tuple(ab0.shape)} in [{ab0.min():.2f}, {ab0.max():.2f}]")

### Print samples

In [ ]:
def show_grid(rows, titles, n=8, start=0):
    fig, axes = plt.subplots(len(rows), n, figsize=(1.3 * n, 1.45 * len(rows)))
    axes = np.atleast_2d(axes)
    for r, (imgs, title) in enumerate(zip(rows, titles)):
        for c in range(n):
            axes[r, c].imshow(imgs[start + c])
            axes[r, c].axis("off")
        axes[r, 0].set_ylabel(title)
        axes[r, 0].axis("on")
        axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    plt.tight_layout()
    plt.show()


gt_rgb = E.lab_to_rgb(val_lab[:8])
gray_rgb = E.reconstruct_rgb(val_lab[:8, ..., 0], np.zeros_like(val_lab[:8, ..., 1:]))
show_grid([gray_rgb, gt_rgb], ["input (L* only)", "ground truth"])

### The grey baseline

The score of trivial answer `a*b* = 0`

In [ ]:
zero_ab = np.zeros((len(val_lab), 2, 32, 32), dtype=np.float32)
baseline = E.evaluate_arrays(zero_ab, val_lab)
print(f"grey baseline on val : {baseline['psnr']:.2f} dB")
print(f"  predicted chroma   : {baseline['pred_chroma']:.2f}   (ground truth {baseline['gt_chroma']:.2f})")
print(f"  colourfulness      : {baseline['pred_colorfulness']:.2f}   (ground truth {baseline['gt_colorfulness']:.2f})")

## Training

### Load model

In [ ]:
from model.cnn import ColorizationCNN

model = ColorizationCNN().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\ntrainable parameters: {n_params:,}")

with torch.no_grad():
    probe = model(torch.zeros(2, 1, 32, 32, device=DEVICE))
assert probe.shape == (2, 2, 32, 32), f"forward returned {tuple(probe.shape)}, expected (2, 2, 32, 32)"

### Parameters setting

In [ ]:
# ============================================================
# parameters settings (tuned, see report)
# ============================================================
EPOCHS = 60
LEARNING_RATE = 1e-3

criterion = nn.L1Loss()

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
# ============================================================

CKPT_PATH = Path("model/ckpt_cnn.pth")   # this is the file you submit
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)

### Training loop

In [ ]:
history = {"train_loss": [], "val_psnr": [], "val_chroma": []}
best_psnr = -float("inf")
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running, seen = 0.0, 0
    for L, ab in train_loader:
        L, ab = L.to(DEVICE, non_blocking=True), ab.to(DEVICE, non_blocking=True)

        # ============================================================
        # optimisation step
        # ============================================================
        optimizer.zero_grad(set_to_none=True)   # 1. clear gradients
        pred = model(L)                         # 2. forward
        loss = criterion(pred, ab)              # 3. loss
        loss.backward()                         # 4. backward
        optimizer.step()                        # 5. optimizer step
        # ============================================================

        running += loss.item() * len(L)
        seen += len(L)

    if scheduler is not None:
        scheduler.step()

    train_loss = running / max(seen, 1)
    val = E.evaluate_model(model, val_lab, device=DEVICE)
    history["train_loss"].append(train_loss)
    history["val_psnr"].append(val["psnr"])
    history["val_chroma"].append(val["pred_chroma"])

    if val["psnr"] > best_psnr:
        best_psnr = val["psnr"]
        torch.save({"state_dict": model.state_dict(), "epoch": epoch,
                    "val_psnr": val["psnr"]}, CKPT_PATH)
        flag = "  <- best"
    else:
        flag = ""
    print(f"epoch {epoch:>3}/{EPOCHS}  loss {train_loss:.4f}  "
          f"val PSNR {val['psnr']:6.3f} dB  chroma {val['pred_chroma']:5.2f}  "
          f"({time.time() - t0:.0f}s){flag}")

print(f"\nbest val PSNR: {best_psnr:.3f} dB   total time: {time.time() - t0:.0f}s")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(history["train_loss"]);  ax[0].set_title("train loss");  ax[0].set_xlabel("epoch")
ax[1].plot(history["val_psnr"])
ax[1].axhline(baseline["psnr"], ls="--", c="grey", label="grey baseline")
ax[1].set_title("val PSNR (dB)"); ax[1].set_xlabel("epoch"); ax[1].legend()
ax[2].plot(history["val_chroma"])
ax[2].axhline(baseline["gt_chroma"], ls="--", c="grey", label="ground truth")
ax[2].set_title("predicted mean chroma"); ax[2].set_xlabel("epoch"); ax[2].legend()
plt.tight_layout(); plt.show()

### Save checkpoint

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()
print(f"restored epoch {ckpt['epoch']} with val PSNR {ckpt['val_psnr']:.3f} dB")

## Testing

### Evaluation

In [ ]:
# test public split 
test_lab = np.load(DATA_DIR / "test_public.npy")        # (N, 32, 32, 3) full Lab
L_test = np.ascontiguousarray(test_lab[..., 0]).astype(np.float32)   # (N, 32, 32) real L*
print(f"test images: {test_lab.shape}")

test_report = E.evaluate_model(model, test_lab, device=DEVICE)
print(f"test PSNR (Base)  : {test_report['psnr']:.3f} dB      <- this is your Base PSNR")
print(f"  base target     : 24.50 dB   ({'reached' if test_report['psnr'] >= 24.5 else 'not yet'})")
print(f"  grey baseline   : 23.49 dB  (gain {test_report['psnr'] - 23.49:+.3f} dB)")
print(f"  mean chroma     : {test_report['pred_chroma']:.2f}  (ground truth {test_report['gt_chroma']:.2f})")


In [ ]:
# here builds the submission file
model.eval()
pred_ab = np.empty((len(L_test), 2, 32, 32), dtype=np.float32)
with torch.no_grad():
    for i in range(0, len(L_test), 256):
        batch = L_test[i:i + 256][:, None] / E.L_SCALE     # (B,1,32,32) normalised
        x = torch.from_numpy(np.ascontiguousarray(batch)).to(DEVICE)
        pred_ab[i:i + 256] = model(x).float().cpu().numpy()

np.save("test_public_pred.npy", pred_ab)
print(f"saved test_public_pred.npy  shape {pred_ab.shape}  dtype {pred_ab.dtype}")
assert pred_ab.shape == (len(L_test), 2, 32, 32) and pred_ab.dtype == np.float32

# The TA re-runs your checkpoint and compares against this file, so it must come from
# the same run as model/ckpt_cnn.pth.  This asserts they already agree.
assert np.allclose(E.psnr_from_ab(pred_ab, test_lab), test_report["psnr"]), \
    "the saved predictions do not match the model you just evaluated"
print("prediction file matches the evaluated model")


### Print samples

In [ ]:
with torch.no_grad():
    Lv, _ = E.normalize_lab(val_lab[:8])
    pv = model(torch.from_numpy(Lv).to(DEVICE)).cpu().numpy()

pred_rgb = E.reconstruct_rgb(val_lab[:8, ..., 0], E.denormalize_ab(pv))
gray_rgb = E.reconstruct_rgb(val_lab[:8, ..., 0], np.zeros_like(val_lab[:8, ..., 1:]))
gt_rgb = E.lab_to_rgb(val_lab[:8])
show_grid([gray_rgb, pred_rgb, gt_rgb], ["input", "prediction", "ground truth"])

In [ ]:
# Best and worst validation images
with torch.no_grad():
    Lv, _ = E.normalize_lab(val_lab)
    preds = np.empty((len(val_lab), 2, 32, 32), dtype=np.float32)
    for i in range(0, len(Lv), 256):
        preds[i:i + 256] = model(torch.from_numpy(Lv[i:i + 256]).to(DEVICE)).cpu().numpy()

per_image = E.psnr_from_ab(preds, val_lab, per_image=True)
order = np.argsort(per_image)
worst, best = order[:8], order[-8:]

for name, idx in (("worst", worst), ("best", best)):
    print(f"{name} 8: PSNR {per_image[idx].min():.2f} - {per_image[idx].max():.2f} dB")
    rgb_pred = E.reconstruct_rgb(val_lab[idx, ..., 0], E.denormalize_ab(preds[idx]))
    show_grid([E.reconstruct_rgb(val_lab[idx, ..., 0], np.zeros_like(val_lab[idx, ..., 1:])),
               rgb_pred, E.lab_to_rgb(val_lab[idx])],
              ["input", "prediction", "ground truth"])

## Submission Reminder

```
Lab02_<student_id>.zip/
└── Lab02_<student_id>/
    ├── model/
    │   ├── cnn.py               # your ColorizationCNN
    │   └── ckpt_cnn.pth         # the weights that produced test_public_pred.npy
    ├── test_public_pred.npy
    └── Lab02.ipynb              # this notebook, run end to end with outputs saved
```